# 2. Analytics Rules and Detection

Analytics rules are the **detection engine** of a SIEM. They're scheduled queries that run against your log data and fire alerts when patterns match.

## Types of Sentinel analytics rules

| Type | How it works | Latency | Use case |
|------|-------------|---------|----------|
| **Scheduled** | KQL query runs on a schedule (e.g., every 5 min) | Minutes | Most detections |
| **NRT (Near Real-Time)** | KQL runs every ~1 minute | ~1 minute | Time-critical threats |
| **Microsoft Security** | Imports alerts from other Defender products | Seconds | XDR correlation |
| **Threat Intelligence** | Matches IOCs against log data | Minutes | Known-bad IPs, domains |
| **Anomaly** | ML-based baseline deviation | Varies | Unusual behavior |
| **Fusion** | Multi-stage attack correlation (ML) | Minutes | Advanced attacks |

In [ ]:
import httpx, json

SIEM = 'http://localhost:8000'

# View existing rules
print('=== Current Analytics Rules ===')
rules = httpx.get(f'{SIEM}/rules').json()
for r in rules:
    sev = {'High': '🔴', 'Medium': '🟡', 'Low': '🟢'}.get(r['severity'], '⬜')
    tactic = r['tactic'] or 'General'
    print(f'  {sev} [{r["id"]}] {r["name"]}')
    print(f'     Table: {r["query_table"]}  |  Tactic: {tactic}  |  Window: {r["window_minutes"]}min  |  Threshold: ≥{r["threshold"]}')
    if r['query_filter']:
        print(f'     Filter: {r["query_filter"]}')
    if r['aggregate_by']:
        print(f'     Group by: {r["aggregate_by"]}')
    print()

In [ ]:
# Create a NEW detection rule — detect privilege escalation
print('=== Creating new analytics rule ===')
r = httpx.post(f'{SIEM}/rules', json={
    'name': 'Suspicious process on database server',
    'severity': 'Critical',
    'tactic': 'PrivilegeEscalation',
    'query_table': 'DeviceEvents',
    'query_filter': {'DeviceName': 'vm-db-01', 'ActionType': 'ProcessCreated'},
    'aggregate_by': 'FileName',
    'threshold': 1,
    'window_minutes': 120,
    'description': 'Any new process on the database server is suspicious — it should only run PostgreSQL.',
})
print(json.dumps(r.json(), indent=2))

In [ ]:
# Run all analytics rules against current data
print('=== Evaluating all analytics rules ===')
r = httpx.post(f'{SIEM}/rules/evaluate')
result = r.json()
print(f'Rules evaluated: {result["evaluated"]}')
print(f'Alerts created: {len(result["alerts_created"])}\n')

for alert in result['alerts_created']:
    print(f'  🚨 {alert["rule"]}: {alert.get("group", "")} ({alert["count"]} events)')

## MITRE ATT&CK mapping

Every analytics rule should map to a **MITRE ATT&CK tactic**. This helps you understand *where in the kill chain* the detection fires:

```
Reconnaissance → Resource Development → Initial Access → Execution → 
Persistence → Privilege Escalation → Defense Evasion → Credential Access → 
Discovery → Lateral Movement → Collection → Command & Control → 
Exfiltration → Impact
```

Our rules map to:

In [ ]:
# MITRE ATT&CK coverage analysis
MITRE_TACTICS = [
    'Reconnaissance', 'ResourceDevelopment', 'InitialAccess', 'Execution',
    'Persistence', 'PrivilegeEscalation', 'DefenseEvasion', 'CredentialAccess',
    'Discovery', 'LateralMovement', 'Collection', 'CommandAndControl',
    'Exfiltration', 'Impact',
]

rules = httpx.get(f'{SIEM}/rules').json()
covered = {r['tactic'] for r in rules if r['tactic']}

print('=== MITRE ATT&CK Coverage Matrix ===\n')
for tactic in MITRE_TACTICS:
    matching = [r['name'] for r in rules if r['tactic'] == tactic]
    if matching:
        print(f'  ✅ {tactic}')
        for name in matching:
            print(f'       └── {name}')
    else:
        print(f'  ⬜ {tactic} — NO DETECTION')

coverage = len(covered) / len(MITRE_TACTICS) * 100
print(f'\nCoverage: {len(covered)}/{len(MITRE_TACTICS)} tactics ({coverage:.0f}%)')
print('\n💡 Real SOCs aim for coverage across all tactics, especially InitialAccess, Execution, and LateralMovement.')

## View generated alerts

Alerts are the output of analytics rules. Each alert has:
- **Severity** — how critical
- **Tactic** — where in the kill chain
- **Entities** — WHO or WHAT is involved
- **Evidence** — sample events that triggered it

In [ ]:
alerts = httpx.get(f'{SIEM}/alerts').json()
print(f'=== All Alerts ({len(alerts)} total) ===\n')
for a in alerts[:10]:
    sev = {'Critical': '🟣', 'High': '🔴', 'Medium': '🟡', 'Low': '🟢'}.get(a['severity'], '⬜')
    print(f'{sev} [{a["status"]}] {a["title"]}')
    if a['entities']:
        entities = json.loads(a['entities']) if isinstance(a['entities'], str) else a['entities']
        print(f'   Entities: {entities}')
    print(f'   Tactic: {a["tactic"]}  |  Created: {a["created_at"]}')
    print()

### SC-200 exam: analytics rule configuration

The exam expects you to configure these fields:

| Field | What it controls |
|-------|-----------------|
| **Query** | KQL that identifies the threat |
| **Query frequency** | How often the rule runs (e.g., every 5 min) |
| **Query lookback** | How far back to search (e.g., last 1 hour) |
| **Trigger threshold** | Minimum result count to fire (e.g., > 0) |
| **Entity mapping** | Which fields are accounts, IPs, hosts |
| **MITRE tactics** | ATT&CK classification |
| **Alert grouping** | Group related alerts (by entity, time, etc.) |
| **Event grouping** | How many events to attach per alert |

**Next**: [Notebook 3 — Incidents and Automation](03_incidents_and_automation.ipynb)